# 00: Storage and shared tables

Creates everything shared by all datasets: the `isic_ml_data_platform` catalog, the `bronze`, `silver` and `gold` schemas, the landing and files Volumes, and the shared tables. Run once per workspace; it's safe to rerun. Dataset-specific tables are created by each dataset's `05` notebook.

In [ ]:
%run ./_setup_env


In [ ]:
from data_platform.labels import SILVER_LABEL_COLUMN_DDL
from data_platform.dataset_layout import load_yaml_config

STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
LANDING_ROOT = STORAGE_CONFIG["landing_root"]
STORAGE_ROOT = STORAGE_CONFIG["storage_root"]

# CATALOG_NAME is already set by _setup_env.ipynb (cell above)
MEDALLION_LAYERS = ["bronze", "silver", "gold"]


In [ ]:
# The one place the catalog is created. CREATE CATALOG IF NOT EXISTS still fails on a workspace
# without a default storage root, even when the catalog exists, so check first.
existing_catalogs = {row.catalog for row in spark.sql("SHOW CATALOGS").collect()}
if CATALOG_NAME not in existing_catalogs:
    spark.sql(f"CREATE CATALOG {CATALOG_NAME}")
spark.sql(f"USE CATALOG {CATALOG_NAME}")

for layer in MEDALLION_LAYERS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{layer}")

spark.sql("CREATE VOLUME IF NOT EXISTS bronze.landing")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze.files")

dbutils.fs.mkdirs(LANDING_ROOT)
dbutils.fs.mkdirs(f"{LANDING_ROOT.rstrip('/')}/archives")

display(spark.sql(f"SHOW VOLUMES IN {CATALOG_NAME}.bronze"))
display(spark.sql("SHOW SCHEMAS"))
display(dbutils.fs.ls(LANDING_ROOT))
display(dbutils.fs.ls(STORAGE_ROOT))

## Shared Bronze table: `bronze.ingestion_runs`

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.ingestion_runs (
      ingestion_run_id STRING,
      dataset_key STRING,
      source_version STRING,
      started_at TIMESTAMP,
      finished_at TIMESTAMP,
      status STRING,
      records_seen INT,
      records_written INT
    )
    USING DELTA
    """
)

display(spark.sql("SHOW TABLES IN bronze"))


## Shared Silver tables

The shared label columns are added separately, because Delta has no `ADD COLUMNS IF NOT EXISTS` ([ADR 002](../docs/decisions/002-silver-label-columns-not-map.md)).

In [ ]:
# Partitioned by dataset_key so several datasets' Silver runs can replace their own rows at the
# same time without their writes conflicting.
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.image_inventory (
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      source_checksum STRING,
      image_width INT,
      image_height INT,
      image_format STRING,
      validation_status STRING,
      validation_reason STRING,
      patient_id STRING,
      lesion_id STRING,
      group_id STRING,
      validated_at TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (dataset_key)
    """
)

existing_columns = {field.name for field in spark.table("silver.image_inventory").schema.fields}
missing_label_columns = [
    f"{name} {ddl}" for name, ddl in SILVER_LABEL_COLUMN_DDL.items() if name not in existing_columns
]
if missing_label_columns:
    spark.sql(f"ALTER TABLE silver.image_inventory ADD COLUMNS ({', '.join(missing_label_columns)})")

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.leakage_groups (
      dataset_key STRING,
      group_id STRING,
      group_type STRING,
      image_count INT
    )
    USING DELTA
    PARTITIONED BY (dataset_key)
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.rejected_records (
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      rejection_reason STRING,
      rejected_at TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (dataset_key)
    """
)

display(spark.sql("SHOW TABLES IN silver"))
display(spark.sql("DESCRIBE TABLE silver.image_inventory"))


## Shared Gold table and view

`gold.manifest_rows` holds every release, keyed by `dataset_version`. `gold.manifest_registry` is a view with one row per release.

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS gold.manifest_rows (
      dataset_version STRING,
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      source_checksum STRING,
      label STRING,
      group_id STRING,
      split STRING,
      sample_seed INT,
      split_seed INT,
      preprocessing_version STRING,
      manifest_row_hash STRING,
      created_at TIMESTAMP,
      git_commit STRING
    )
    USING DELTA
    """
)
# Adds git_commit to a table created by an older version of this notebook.
if "git_commit" not in {field.name for field in spark.table("gold.manifest_rows").schema.fields}:
    spark.sql("ALTER TABLE gold.manifest_rows ADD COLUMNS (git_commit STRING)")

spark.sql(
    """
    CREATE OR REPLACE VIEW gold.manifest_registry AS
    SELECT
      dataset_version,
      sort_array(collect_set(dataset_key)) AS dataset_keys,
      COUNT(*) AS image_count,
      SUM(CASE WHEN split = 'train' THEN 1 ELSE 0 END) AS train_count,
      SUM(CASE WHEN split = 'validation' THEN 1 ELSE 0 END) AS validation_count,
      SUM(CASE WHEN split = 'test' THEN 1 ELSE 0 END) AS test_count,
      sort_array(collect_set(label)) AS label_values,
      sort_array(collect_set(preprocessing_version)) AS preprocessing_versions,
      sort_array(collect_set(sample_seed)) AS sample_seeds,
      sort_array(collect_set(split_seed)) AS split_seeds,
      MIN(created_at) AS first_created_at,
      MAX(created_at) AS last_updated_at,
      sort_array(collect_set(git_commit)) AS git_commits
    FROM gold.manifest_rows
    GROUP BY dataset_version
    """
)

display(spark.sql("SHOW TABLES IN gold"))
display(spark.sql("DESCRIBE TABLE gold.manifest_rows"))
display(spark.sql("SELECT * FROM gold.manifest_registry"))
